In [245]:
import sqlite3
import pandas as pd
import numpy as np

In [246]:
conn = sqlite3.connect(
    "../data/raw/eicu_v2_0_1.sqlite3"
)

In [247]:
pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM patient;",
    conn
)

,n
0,2520


In [248]:
patient_core = pd.read_sql_query(
    """
    SELECT
        patientunitstayid,
        patienthealthsystemstayid,
        gender,
        age,
        ethnicity,
        hospitalid,
        wardid,
        apacheadmissiondx,
        hospitaladmitsource,
        hospitaldischargestatus,
        hospitaldischargelocation,
        unittype,
        unitadmitsource,
        unitstaytype,
        unitdischargestatus,
        unitdischargelocation,
        unitdischargeoffset
    FROM patient;
    """,
    conn
)

patient_core["los_days"] = (
    patient_core["unitdischargeoffset"] / 1440
)

patient_core["icu_death"] = (
    patient_core["unitdischargestatus"]
    .eq("Expired")
    .astype(int)
)

patient_core.shape

(2520, 19)

In [249]:
medication_summary = pd.read_sql_query(
    """
    SELECT
        patientunitstayid,
        COUNT(*) AS n_medication_records
    FROM medication
    GROUP BY patientunitstayid;
    """,
    conn
)

In [250]:
lab_summary = pd.read_sql_query(
    """
    SELECT
        patientunitstayid,
        COUNT(*) AS n_lab_records
    FROM lab
    GROUP BY patientunitstayid;
    """,
    conn
)

In [251]:
apache_summary = pd.read_sql_query(
    """
    SELECT
        patientunitstayid,
        apachescore AS apache_score,
        predictedicumortality AS predicted_icu_mortality,
        predictediculos AS predicted_icu_los,
        actualventdays AS actual_vent_days,
        apacheversion AS apache_version
    FROM apachePatientResult
    WHERE apacheversion = 'IVa';
    """,
    conn
)

In [252]:
numeric_apache_columns = [
    "apache_score",
    "predicted_icu_mortality",
    "predicted_icu_los",
    "actual_vent_days"
]

for col in numeric_apache_columns:
    apache_summary[col] = pd.to_numeric(
        apache_summary[col],
        errors="coerce"
    )

In [253]:
apache_summary[numeric_apache_columns] = (
    apache_summary[numeric_apache_columns]
    .replace(-1, np.nan)
)

In [254]:
apache_summary["ventilation_status"] = np.where(
    apache_summary["actual_vent_days"] > 0,
    "Ventilated",
    "No ventilation data"
)

In [255]:
apache_summary["ventilation_data_available"] = (
    apache_summary["actual_vent_days"]
    .notna()
    .astype(int)
)

In [256]:
apache_summary[
    "ventilation_status"
].value_counts()

ventilation_status
No ventilation data    1317
Ventilated              521
Name: count, dtype: int64

In [257]:
print(
    "Duplicados APACHE:",
    apache_summary[
        "patientunitstayid"
    ].duplicated().sum()
)

Duplicados APACHE: 0


In [258]:
diagnosis_summary = pd.read_sql_query(
    """
    SELECT
        patientunitstayid,
        COUNT(*) AS n_diagnoses
    FROM diagnosis
    GROUP BY patientunitstayid;
    """,
    conn
)

In [259]:
diagnosis_summary.head()

,patientunitstayid,n_diagnoses
0,143870,3
1,145427,8
2,151179,18
3,151867,2
4,151900,31


In [260]:
medication_summary = pd.read_sql_query(
    """
    SELECT
        patientunitstayid,
        COUNT(*) AS n_medication_records
    FROM medication
    GROUP BY patientunitstayid;
    """,
    conn
)

In [261]:
lab_summary = pd.read_sql_query(
    """
    SELECT
        patientunitstayid,
        COUNT(*) AS n_lab_records
    FROM lab
    GROUP BY patientunitstayid;
    """,
    conn
)

In [262]:
print(patient_core.shape)
print(diagnosis_summary.shape)
print(medication_summary.shape)
print(lab_summary.shape)
print(apache_summary.shape)

(2520, 19)
(2155, 2)
(1857, 2)
(2444, 2)
(1838, 8)


In [263]:
icu_analytics_v2 = (
    patient_core
    .merge(
        diagnosis_summary,
        on="patientunitstayid",
        how="left"
    )
    .merge(
        medication_summary,
        on="patientunitstayid",
        how="left"
    )
    .merge(
        lab_summary,
        on="patientunitstayid",
        how="left"
    )
    .merge(
        apache_summary,
        on="patientunitstayid",
        how="left"
    )
)

In [264]:
print("Shape:", icu_analytics_v2.shape)

print(
    "Duplicados:",
    icu_analytics_v2["patientunitstayid"].duplicated().sum()
)

Shape: (2520, 29)
Duplicados: 0


In [265]:
print("Linhas:", icu_analytics_v2.shape[0])
print(
    "Duplicados:",
    icu_analytics_v2["patientunitstayid"].duplicated().sum()
)
print(
    "LOS médio:",
    icu_analytics_v2["los_days"].mean()
)
print(
    "Mortalidade:",
    icu_analytics_v2["icu_death"].mean()
)

Linhas: 2520
Duplicados: 0
LOS médio: 2.419496527777778
Mortalidade: 0.05


In [266]:
icu_analytics_v2.to_csv(
    "../data/processed/icu_analytics_v2.csv",
    index=False
)